# 8주차 과제 — 질문 하나에서 시작하는 미니 프로젝트

이번 주에는 프로젝트를 끝까지 완성하는 것보다 **시도·오류·문제 분해 기록**을 남기는 일이 더 중요합니다. 아래 세 가지 트랙 중 관심 있는 하나를 선택합니다. 첫 코드 셀에서 `TRACK`을 고르고 `LOAD_DATA = False`로 두면, 원본 파일이 없어도 경로 점검부터 위에서 차례로 실행할 수 있습니다.

- **트랙 A (제공)**: [제주도교통량](https://dacon.io/competitions/official/235985) — 아래에 샘플링 코드를 제공합니다.
- **트랙 B (제공)**: [신용카드세그먼트](https://dacon.io/competitions/official/236460) — 아래에 테이블 조인 예시를 제공합니다.
- **트랙 C (자유)**: 원하는 주제 + 직접 수집한 데이터 — 베이스라인 없음

- ✅ **기본 시도**: 질문 하나, 실행 또는 실행 시도 하나, 관찰 또는 오류 하나, 다음 행동 하나를 기록합니다.
- 🧩 **문제 분해**: 막힌 지점을 파일·경로·메모리·자료형·질문 중 하나로 좁힙니다.
- 🌱 **선택 탐색**: EDA·조인 검산·모델·그래프 가운데 한 가지를 더 진행합니다.

`example.ipynb`의 체크리스트를 먼저 살펴보면 시작점을 정하는 데 도움이 됩니다. **기본 완료 기준**은 질문 1개 + 실행 또는 실행 시도 1개 + 관찰한 결과 또는 오류 1개 + 다음 행동 1개입니다. 분석 성공 여부는 기본 완료 조건이 아닙니다.


## 트랙 A — 제주도 도로 교통량

도로 구간별 통행 속도(`target`)를 예측·분석하는 데이터입니다. [제주도교통량 원본 페이지](https://dacon.io/competitions/official/235985)에서 데이터 설명과 이용 조건을 확인하고 파일을 내려받을 수 있습니다. 별도의 3만 행 샘플 파일은 제공되지 않습니다. `LOAD_DATA = True`로 바꾸면 원본 약 470만 행을 먼저 메모리에 불러온 뒤 3만 행을 무작위로 추출합니다. 메모리가 부족하면 같은 셀을 반복 실행하지 말고 오류를 기록한 뒤 환경 정보와 함께 DM으로 문의합니다.


In [ ]:
from pathlib import Path
import pandas as pd

TRACK = 'A'       # 'A', 'B', 'C' 중 하나를 선택합니다.
LOAD_DATA = False  # 파일을 준비한 뒤 True로 바꿉니다.
CUSTOM_PATH = ''   # 트랙 C라면 파일 경로를 적습니다.

if TRACK not in {'A', 'B', 'C'}:
    raise ValueError("TRACK은 'A', 'B', 'C' 중 하나여야 합니다.")

jeju_path = Path('../dataset/extracted/제주도 도로 교통량 예측 AI 경진대회/open/train.csv')
member_path = Path('../dataset/extracted/신용카드 고객 세그먼트 분류 AI 경진대회/train/1.회원정보/201807_train_회원정보.parquet')
credit_path = Path('../dataset/extracted/신용카드 고객 세그먼트 분류 AI 경진대회/train/2.신용정보/201807_train_신용정보.parquet')

if TRACK == 'A':
    paths_to_check = [('제주 train.csv', jeju_path)]
elif TRACK == 'B':
    paths_to_check = [('회원정보', member_path), ('신용정보', credit_path)]
else:
    custom_path = Path(CUSTOM_PATH) if CUSTOM_PATH.strip() else None
    paths_to_check = [('자유 데이터', custom_path)]

path_check = pd.DataFrame([
    {'파일': name, '예상 경로': str(path) if path else '(아직 입력하지 않음)', '존재 여부': bool(path and path.is_file())}
    for name, path in paths_to_check
])
print('선택한 트랙:', TRACK, '/ 실제 데이터 로딩:', LOAD_DATA)
display(path_check)

jeju_sample = None
member = credit = joined = analysis_data = custom_data = None
load_status = '파일 존재 여부만 확인했습니다. LOAD_DATA는 False입니다.'

if TRACK == 'A' and LOAD_DATA:
    if not jeju_path.is_file():
        load_status = f'파일을 찾지 못했습니다: {jeju_path}'
    else:
        try:
            jeju = pd.read_csv(jeju_path)
            print('원본:', jeju.shape)
            jeju_sample = jeju.sample(n=30_000, random_state=42).reset_index(drop=True)
            del jeju
            load_status = f'제주 표본을 준비했습니다: {jeju_sample.shape}'
        except Exception as error:
            load_status = f'{type(error).__name__}: {error}'
    print(load_status)
elif TRACK == 'C' and LOAD_DATA:
    if custom_path is None or not custom_path.is_file():
        load_status = f'자유 데이터 파일을 찾지 못했습니다: {CUSTOM_PATH or "경로 미입력"}'
    else:
        try:
            if custom_path.suffix.lower() == '.csv':
                custom_data = pd.read_csv(custom_path)
            elif custom_path.suffix.lower() in {'.parquet', '.pq'}:
                custom_data = pd.read_parquet(custom_path)
            elif custom_path.suffix.lower() in {'.xlsx', '.xls'}:
                custom_data = pd.read_excel(custom_path)
            else:
                raise ValueError('CSV, Parquet, Excel 파일만 기본 로더가 지원합니다.')
            load_status = f'자유 데이터를 불러왔습니다: {custom_data.shape}'
        except Exception as error:
            load_status = f'{type(error).__name__}: {error}'
    print(load_status)


`start_latitude`/`start_longitude`처럼 위경도 컬럼이 있어 7주차에서 다룬 위치 기반 분석과도 연결할 수 있습니다. 예를 들어 특정 도로 구간의 정체 패턴을 지도에 표시할 수 있습니다. 먼저 `road_name`, `base_hour`, `day_of_week`별로 속도가 어떻게 다른지 살펴보면 데이터의 구조를 이해하는 데 도움이 됩니다.


## 트랙 B — 신용카드 고객 세그먼트

[신용카드세그먼트 원본 페이지](https://dacon.io/competitions/official/236460)에서 데이터 설명과 이용 조건을 확인하고 파일을 내려받을 수 있습니다. 데이터는 월별 스냅샷 형태이며, 8개 영역(회원정보/신용정보/승인매출정보/청구입금정보/잔액정보/채널정보/마케팅정보/성과정보)으로 나뉩니다. `LOAD_DATA = True`로 바꾸면 회원정보와 신용정보 두 영역의 2018년 7월 자료만 조인합니다. Parquet 엔진 오류가 나면 오류 문구를 먼저 기록한 뒤 터미널에서 `python -m pip install pyarrow`를 실행합니다.


In [ ]:
if TRACK == 'B' and LOAD_DATA:
    if not member_path.is_file() or not credit_path.is_file():
        load_status = '회원정보 또는 신용정보 파일을 찾지 못했습니다. 위 경로 점검표를 확인합니다.'
    else:
        try:
            member = pd.read_parquet(member_path)
            credit = pd.read_parquet(credit_path)
            load_status = f'회원정보 {member.shape}, 신용정보 {credit.shape}를 불러왔습니다.'
        except Exception as error:
            load_status = f'{type(error).__name__}: {error}'
    print(load_status)
elif TRACK == 'B':
    print('경로만 확인했습니다. 파일을 준비한 뒤 LOAD_DATA = True로 바꿉니다.')
else:
    print('트랙 B를 선택하지 않아 Parquet 파일을 불러오지 않습니다.')


두 테이블에는 각각 78개와 42개의 컬럼이 있어 전체를 조인하면 실습 범위가 커질 수 있습니다. 여기서는 분석에 필요한 일부 컬럼을 선택해 조인 과정을 살펴봅니다.


In [ ]:
if TRACK == 'B' and member is not None and credit is not None:
    member_cols = ['기준년월', 'ID', '남녀구분코드', '연령', 'Segment', 'Life_Stage']
    credit_cols = ['기준년월', 'ID', '최초한도금액', '카드이용한도금액', 'CA한도금액']
    join_keys = ['기준년월', 'ID']

    print('회원정보 중복 키:', member.duplicated(join_keys).sum())
    print('신용정보 중복 키:', credit.duplicated(join_keys).sum())
    try:
        joined = member[member_cols].merge(
            credit[credit_cols], on=join_keys, how='left',
            validate='one_to_one', indicator=True,
        )
        print('조인 전후 행 수:', len(member), '→', len(joined))
        print('매칭 결과:', joined['_merge'].value_counts().to_dict())
        joined = joined.drop(columns='_merge')
        load_status = f'두 테이블을 조인했습니다: {joined.shape}'
    except Exception as error:
        joined = None
        load_status = f'조인 오류 — {type(error).__name__}: {error}'
        print(load_status)
elif TRACK == 'B':
    print('불러온 두 테이블이 없어 조인을 건너뜁니다. 현재 상태:', load_status)
else:
    print('트랙 B 조인을 건너뜁니다.')


In [ ]:
if TRACK == 'B' and joined is not None:
    print(joined['Segment'].value_counts())
elif TRACK == 'B':
    print('조인 데이터가 없어 클래스 분포 확인을 건너뜁니다.')
else:
    print('트랙 B 클래스 분포 확인을 건너뜁니다.')


`Segment`는 이 경진대회의 예측 대상입니다. 2018년 7월에는 A 162명, B 24명인 반면 E는 320,342명입니다. 5천 행 비율 표본을 만들면 A/B가 거의 남지 않아 클래스별 지표를 계산하기 어렵습니다. 따라서 기본 코드에서는 40만 행을 유지하며, 크기를 줄이는 실험은 희소 클래스를 보존하는 방법과 표본 분포의 한계를 정한 뒤 선택적으로 수행합니다.


In [ ]:
if TRACK == 'B' and joined is not None:
    analysis_data = joined  # 희소한 A/B를 보존하기 위해 기본 시도에서는 전체를 사용합니다.
    print('분석 데이터:', analysis_data.shape)
    print(analysis_data['Segment'].value_counts().to_dict())
elif TRACK == 'B':
    print('분석 데이터가 아직 없습니다. 경로 점검과 오류 기록부터 진행합니다.')
else:
    print('트랙 B 분석 데이터 준비를 건너뜁니다.')


## 트랙 C — 자유 주제

이 저장소 밖의 데이터를 직접 찾아 사용할 수도 있습니다. 첫 코드 셀의 `CUSTOM_PATH`에 CSV·Parquet·Excel 파일 경로를 적습니다. 파일이 아직 없다면 경로를 비워 둔 채 질문과 데이터 요구사항부터 기록합니다. 자유 주제에는 정해진 분석 베이스라인이 없으므로 `example.ipynb` Part 1의 문제 정의와 Part 2~3의 키 매칭·단위 통일 점검을 시작점으로 사용합니다.


## ✅ 기본 시도 — 질문 하나와 첫 관찰을 만듭니다

아래 셀은 데이터가 있으면 작은 요약표를 만들고, 데이터가 없으면 앞에서 만든 경로 점검표를 결과로 사용합니다. 따라서 파일을 불러오지 못했어도 그대로 실행할 수 있습니다.


In [ ]:
question_by_track = {
    'A': '요일별 평균 통행속도는 어떻게 다릅니까?',
    'B': '2018년 7월 고객 Segment의 클래스 비율은 어떻게 다릅니까?',
    'C': '이 데이터의 한 행은 무엇이며 가장 먼저 확인할 품질 문제는 무엇입니까?',
}
question = question_by_track[TRACK]

if TRACK == 'A' and jeju_sample is not None:
    analysis_preview = (
        jeju_sample.groupby('day_of_week')['target']
        .agg(['count', 'mean']).round(2).sort_values('mean')
    )
    slowest_day = analysis_preview['mean'].idxmin()
    attempt = '3만 행 표본에서 요일별 target 개수와 평균을 계산했습니다.'
    observation = f'표본에서 평균 통행속도가 가장 낮은 요일은 {slowest_day}입니다.'
    next_action = '요일별 표본 수와 원본 대표성의 한계를 확인하겠습니다.'
elif TRACK == 'B' and analysis_data is not None:
    analysis_preview = (
        analysis_data['Segment'].value_counts(dropna=False)
        .rename_axis('Segment').reset_index(name='고객 수')
    )
    attempt = '2018년 7월 Segment별 고객 수를 계산했습니다.'
    observation = f'가장 많은 클래스는 {analysis_preview.loc[0, "Segment"]}이며 고객 수는 {analysis_preview.loc[0, "고객 수"]:,}명입니다.'
    next_action = '가장 적은 클래스의 표본 수를 확인하고 사용할 평가 지표를 정하겠습니다.'
elif TRACK == 'C' and custom_data is not None:
    analysis_preview = pd.DataFrame([{
        '행 수': len(custom_data),
        '열 수': custom_data.shape[1],
        '전체 결측 셀 수': int(custom_data.isna().sum().sum()),
    }])
    attempt = '자유 데이터의 행·열 수와 전체 결측 셀 수를 계산했습니다.'
    observation = f'데이터는 {custom_data.shape[0]:,}행, {custom_data.shape[1]:,}열입니다.'
    next_action = '한 행의 의미와 결측치가 많은 컬럼 하나를 확인하겠습니다.'
else:
    analysis_preview = path_check.copy()
    attempt = '선택한 트랙의 예상 파일 경로와 존재 여부를 확인했습니다.'
    observation = load_status
    if bool(path_check['존재 여부'].all()):
        next_action = '컴퓨터의 메모리와 패키지를 확인한 뒤 LOAD_DATA = True로 바꾸겠습니다.'
    elif TRACK == 'C':
        next_action = '사용할 데이터의 출처를 정하고 CUSTOM_PATH에 실제 경로를 적겠습니다.'
    else:
        next_action = '원본 페이지에서 파일을 내려받아 예상 경로에 배치하겠습니다.'

print('질문:', question)
print('실행 또는 실행 시도:', attempt)
print('관찰:', observation)
print('다음 행동:', next_action)
analysis_preview


작은 힌트: 기본 질문이 마음에 들지 않으면 `question` 문장만 바꾸어도 됩니다. 데이터가 없을 때는 파일 부재가 관찰 결과이고, 다운로드·경로 수정·환경 확인 가운데 하나가 다음 행동입니다.


## 🧩 문제 분해 — 어디에서 멈췄는지 좁힙니다

아래 표에서 현재 상태와 가까운 행 하나를 고릅니다. 한 번에 모든 문제를 해결할 필요는 없습니다.


In [ ]:
breakdown = pd.DataFrame([
    {'단계': '파일', '확인 질문': '예상 경로에 파일이 있습니까?', '현재 관찰': '있음' if bool(path_check['존재 여부'].all()) else '없거나 경로 미입력'},
    {'단계': '로딩', '확인 질문': '실제 로딩을 시도했습니까?', '현재 관찰': '시도함' if LOAD_DATA else '경로만 확인함'},
    {'단계': '환경', '확인 질문': '메모리·Parquet 엔진 오류가 있습니까?', '현재 관찰': load_status},
    {'단계': '질문', '확인 질문': '한 문장 질문을 정했습니까?', '현재 관찰': question},
    {'단계': '다음 행동', '확인 질문': '가장 작은 다음 행동은 무엇입니까?', '현재 관찰': next_action},
])
breakdown


## ✅ 이번 주 학습 기록

앞에서 자동으로 만든 네 줄을 확인하고, 실제 경험과 다른 문장이 있으면 해당 변수의 내용을 고쳐 다시 실행합니다. 오류가 있었다면 전체 화면 대신 오류 종류와 핵심 문구를 `observation`에 남깁니다.


In [ ]:
learning_log = pd.DataFrame({
    '기록 항목': ['질문', '실행 또는 실행 시도', '관찰한 결과 또는 오류', '다음 행동'],
    '내용': [question, attempt, observation, next_action],
})
learning_log


> **여기까지 하면 이번 주 기록 완료입니다.** 네 줄 표를 1~2장의 발표 자료에 옮겨 시도와 브레이크다운을 공유합니다. 모델·최종 그래프·완성된 결론은 기본 완료 조건이 아닙니다.


## 🌱 선택 탐색 — 요약을 하나 더 만듭니다

데이터를 불러온 경우에만 아래 셀이 추가 요약을 만듭니다. 출력이 없어도 기본 기록에는 영향이 없습니다.


In [ ]:
if TRACK == 'A' and jeju_sample is not None:
    optional_result = jeju_sample.groupby('base_hour')['target'].agg(['count', 'mean']).round(2)
    print('시간대별 통행속도 요약입니다.')
    display(optional_result.head())
elif TRACK == 'B' and analysis_data is not None:
    optional_result = analysis_data['Segment'].value_counts(normalize=True).mul(100).round(3)
    print('Segment별 비율(%)입니다.')
    display(optional_result)
elif TRACK == 'C' and custom_data is not None:
    optional_result = custom_data.isna().mean().mul(100).sort_values(ascending=False).head(10).round(2)
    print('결측 비율이 높은 컬럼 상위 10개입니다.')
    display(optional_result)
else:
    print('불러온 데이터가 없어 선택 탐색을 건너뜁니다. 기본 학습 기록은 이미 완료할 수 있습니다.')


## 🌱 프로젝트로 더 발전시키고 싶다면

관심이 있다면 다음 가운데 한 가지만 더 선택합니다.

- 질문에 직접 답하는 그래프나 표를 1개 만듭니다.
- 단순 기준선과 비교할 모델을 1개 만듭니다.
- 조인 전후 행 수·중복 키·결측치를 검산합니다.
- 데이터의 기간·표본·대표성 한계를 한 문장으로 적습니다.
- 다음 발표 구조를 사용합니다: 질문 → 시도 → 근거 또는 오류 → 배운 점 → 다음 행동.

여러 항목을 모두 수행할 필요는 없습니다. 새로운 시도에서 오류가 나면 그 오류도 다음 학습 기록으로 사용할 수 있습니다.
